# Monitoring a served model

Once a model is [serving traffic](../08-persistence-deployment/serving-a-model.ipynb),
you have to watch it. Three **distinct** concerns, easily conflated:

1. **Operational health** — latency, error rate, request volume. Is the server
   up and responding?
2. **Data drift** — are incoming request features statistically different from
   the training distribution?
3. **Model/prediction drift** — is the distribution of predictions shifting, or
   (once ground truth arrives) is accuracy degrading?

We cover operational health with the general-purpose
[`tracing`](https://docs.rs/tracing) + [`metrics`](https://docs.rs/metrics)
crates, and data / prediction drift with
[`driftwatch`](https://crates.io/crates/driftwatch), a purpose-built drift
detector.

## Operational health: structured logging & metrics

Instrument each request with [`tracing`](https://docs.rs/tracing) (structured
logs) and [`metrics`](https://docs.rs/metrics) (counters/histograms). In the
[axum server](../08-persistence-deployment/serving-a-model.ipynb) this goes in
the handler; here we emit the same signals directly:

In [ ]:
:dep tracing = { version = "0.1" }
:dep tracing-subscriber = { version = "0.3" }
:dep metrics = { version = "0.24" }

tracing_subscriber::fmt::init();

// Per-request: a structured log line (request id, latency, prediction) ...
tracing::info!(request_id = 42, latency_ms = 8.1, prediction = 0.87, "handled /predict");
// ... and metrics a /metrics endpoint (via the prometheus exporter) would expose.
metrics::counter!("predict_requests_total").increment(1);
metrics::histogram!("predict_latency_ms").record(8.1);
println!("logged one request and recorded its metrics");

In production you'd install a Prometheus recorder (`metrics-exporter-prometheus`)
and expose a `GET /metrics` route alongside `/predict`; Prometheus scrapes it and
you alert on latency/error thresholds.

## Data drift: Population Stability Index

**PSI** compares a live window of a feature against its training distribution,
bin by bin. [`driftwatch`](https://crates.io/crates/driftwatch) computes it — plus
KL / JS divergence and a KS two-sample test — and rolls the per-feature results
into a `DriftReport` with an overall verdict. You `fit_continuous` a
`ReferenceDistribution` from training data, add it to a `DatasetMonitor`, then
`check` each live batch:

In [ ]:
:dep driftwatch = { version = "0.1.0" }
use driftwatch::{DatasetMonitor, EqualFrequencyBinning, LiveFeature, ReferenceDistribution};

// A tiny deterministic PRNG so we need no `rand` dependency (crude normal-ish
// samples: the mean of three uniforms).
struct Lcg(u64);
impl Lcg {
    fn unit(&mut self) -> f64 { self.0 = self.0.wrapping_mul(6364136223846793005).wrapping_add(1442695040888963407); (self.0 >> 11) as f64 / (1u64 << 53) as f64 }
    fn sample(&mut self, mean: f64, spread: f64) -> f64 { let u = self.unit() + self.unit() + self.unit(); mean + (u - 1.5) * spread }
}

// The `amount` request feature's distribution when the model was trained.
let ref_amount: Vec<f64> = { let mut r = Lcg(0x51ce); (0..1000).map(|_| r.sample(50.0, 10.0)).collect() };
println!("reference built from {} training samples", ref_amount.len());

In [ ]:
{
    // A DatasetMonitor holds one ReferenceDistribution per feature. PSI is the
    // default per-feature metric (drift threshold 0.25); it also runs a KS
    // two-sample test, and flags the whole dataset once enough features drift.
    let mut monitor = DatasetMonitor::new();
    monitor.add_feature(ReferenceDistribution::fit_continuous("amount", &ref_amount, EqualFrequencyBinning::default()).unwrap());

    // Two rolling windows of live requests: one on-distribution, one shifted up.
    let mut r = Lcg(0xd21f);
    let live_stable: Vec<f64> = (0..500).map(|_| r.sample(50.0, 10.0)).collect();
    let live_drift:  Vec<f64> = (0..500).map(|_| r.sample(80.0, 12.0)).collect();

    let stable = monitor.check(&[("amount", LiveFeature::Continuous(&live_stable))]).unwrap();
    let drift  = monitor.check(&[("amount", LiveFeature::Continuous(&live_drift))]).unwrap();
    println!("STABLE window\n{stable}");
    println!("DRIFTED window\n{drift}");
}

The stable window's PSI sits near zero and its verdict reads *stable*; the shifted
window trips the drift threshold and the dataset verdict flips to *DRIFTED* — a
signal to investigate.

## What to do when drift fires

Detection is the easy part; the response is a judgement call. The usual options:
**alert** a human, **shift traffic** gradually to a challenger model, or trigger
**scheduled retraining**. `driftwatch` covers the alerting side — attach a
`LogAlerter` (structured `tracing` events) or a `WebhookAlerter` (POST to
Slack/PagerDuty) with `.with_alerter(...)`, firing only when the aggregate verdict
crosses; it also ships a live HTTP drift dashboard and Prometheus gauge export. A
full retraining pipeline is a natural next step, not something we build here.

```{note}
**Ecosystem maturity.** Rust's MLOps-monitoring story is younger than Python's
(Evidently, WhyLabs) but no longer bare:
[`driftwatch`](https://crates.io/crates/driftwatch) provides the drift detection
(PSI, KL/JS, KS/chi-square, live windowing, alerting, a dashboard), while
operational health still leans on the general-purpose `tracing` / `metrics` /
`axum` stack. It's newer and single-author, so re-check versions before relying
on it.
```

Next: [Time Series](../10-time-series/time-series-fundamentals.ipynb) — a
different data shape, where ordering itself carries information.